# NPCC simulation study

This notebook drives the packaged simulation study in `npcc.experiments`.
It sweeps a grid over copula **families**, conditional / unconditional
**Kendall's-tau scenarios**, support **transforms**, **normalization**
(Sinkhorn) on/off, and the **criterion vs quantiles** density-recovery
methods, scoring each fit against the exact pyvinecopulib ground truth.

For large runs use the CLI instead and load the saved tables here:

```bash
uv run npcc-simstudy --config ../configs/study.toml --out ../results --workers 4
```

Requires the `experiments` extra (`pandas`, `matplotlib`) and a TabPFN token.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv

from npcc.experiments import GridConfig, RunConfig, aggregate_results, run_study
from npcc.experiments import plots

load_dotenv()  # picks up TABPFN_TOKEN from .env

## Define the grid

A small grid that runs in a few minutes. Scale it up here or via the TOML
config consumed by the CLI.

In [ ]:
grid = GridConfig(
    families=["clayton", "gumbel"],
    tau_scenarios=["linear", "uncond50"],
    transforms=["logit", "identity"],
    methods=["criterion", "quantiles"],
    normalize=[None, 5],
    n=[200, 500],
    n_rep=2,
    projection_grid_size=30,
)
run = RunConfig(out=Path("../results"), workers=1)

## Run the study

(Or skip this and load `metrics.csv` / `runtime.csv` written by the CLI.)

In [ ]:
metrics_df, runtime_df, wall = run_study(grid, run)
print(f"wall: {wall:.1f}s, {len(metrics_df)} metric rows")
mc_summary, runtime_summary = aggregate_results(metrics_df, runtime_df)
mc_summary.head()

## Compare across axes

Pick a `(family, tau_scenario, quantity, metric)` slice and compare across any
axis via `hue` / `by` (`method`, `transform`, `normalize`, ...).

In [ ]:
plots.mean_metric_lineplot(
    mc_summary,
    family="clayton",
    tau_scenario="linear",
    quantity="pdf",
    metric="ISE",
    hue="method",
);

In [ ]:
plots.metric_boxplot(
    metrics_df,
    family="clayton",
    tau_scenario="linear",
    quantity="pdf",
    metric="KL",
    n=500,
    by="transform",
);